# 01 · Bronze — Ingesta raw del dataset NIES

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Ingestar el fichero Excel del NIES (`2_water quality_eng_v20250925.xlsx`) y almacenarlo
en formato **Delta Lake** como capa Bronze, con metadatos de procedencia añadidos.

## Estructura del notebook
1. Configuración del entorno
2. Lectura y normalización del Excel NIES
3. Conversión a Spark y enriquecimiento con metadatos
4. Escritura en Delta Lake (capa Bronze)
5. Validación de la ingesta

## 1. Configuración del entorno

In [0]:
# ──────────────────────────────────────────────────────────────────────────────
# Celda 1 — Importaciones y variables de configuración
# ──────────────────────────────────────────────────────────────────────────────
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, IntegerType
)
from datetime import datetime

# ── Parámetros de configuración ──────────────────────────────────────────────
NIES_EXCEL_PATH = "/FileStore/tables/2_water_quality_eng_v20250925.xlsx"
# En Databricks Community Edition el fichero debe subirse con:
#   Data → Add Data → Upload File → luego usar la ruta /FileStore/tables/

CATALOG = "training"            # Catálogo Unity Catalog (detección automática)
SCHEMA  = "kasumigaura"         # Esquema (base de datos)
TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"

EXCEL_HEADER_ROW = 2            # El fichero NIES tiene 2 filas de encabezado
INGESTION_TIMESTAMP = datetime.utcnow().isoformat()
DATASET_VERSION = "v20250925"   # Versión del fichero NIES
UC_SOURCE_TABLE = 'training.default.2_water_quality_eng_v_20250925'

print(f"Tabla destino : {TABLE_BRONZE}")
print(f"Timestamp     : {INGESTION_TIMESTAMP}")
print(f"Versión NIES  : {DATASET_VERSION}")

Tabla destino : training.kasumigaura.bronze_water_quality
Timestamp     : 2026-11-25T02:32:34.738058
Versión NIES  : v20250925


/home/spark-caed0e06-434f-47bc-85a4-b3/.ipykernel/79/command-6514421613040311-305715966:22: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  INGESTION_TIMESTAMP = datetime.utcnow().isoformat()


In [0]:
# Celda 2 — Verificar accesibilidad del fichero NIES en DBFS
import os

dbfs_path = NIES_EXCEL_PATH.replace("/FileStore", "/dbfs/FileStore")
UC_SOURCE_TABLE = "training.default.2_water_quality_eng_v_20250925"
USE_UC_TABLE = False

if os.path.exists(dbfs_path):
    size_mb = os.path.getsize(dbfs_path) / 1_048_576
    print(f"✓ Fichero Excel encontrado en DBFS: {dbfs_path}")
    print(f"  Tamaño: {size_mb:.2f} MB")
else:
    try:
        spark.table(UC_SOURCE_TABLE).limit(1).collect()
        USE_UC_TABLE = True
        print(f"✓ Tabla Unity Catalog encontrada: {UC_SOURCE_TABLE}")
        print(f"  → El notebook leerá desde esta tabla en lugar del Excel")
    except Exception as e:
        raise FileNotFoundError(f"No se encontró el dataset NIES: {e}")

✓ Tabla Unity Catalog encontrada: training.default.2_water_quality_eng_v_20250925
  → El notebook leerá desde esta tabla en lugar del Excel


In [0]:
# Celda 3 — Detectar catálogo Unity Catalog activo
try:
    active_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    print(f"Catálogo activo: {active_catalog}")
    if active_catalog != CATALOG:
        print(f"  → Usando catálogo detectado: {active_catalog}")
        CATALOG = active_catalog
        TABLE_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_water_quality"
        print(f"  → Tabla ajustada: {TABLE_BRONZE}")
except Exception as e:
    print(f"No se puede detectar catálogo ({e}). Usando: {CATALOG}")

Catálogo activo: workspace
  → Usando catálogo detectado: workspace
  → Tabla ajustada: workspace.kasumigaura.bronze_water_quality


## 2. Lectura y normalización del Excel NIES

In [0]:
# Celda 4 — Leer el dataset NIES
import os
import pandas as pd

EXCEL_PATH = "/Workspace/Users/kathianagarcia27@gmail.com/kasumigaura-lakehouse-poc/data/2_water quality_eng_v20250925.xlsx"

df_pandas = pd.read_excel(EXCEL_PATH, sheet_name="water quality", header=0, engine="openpyxl")
print(f"✓ Shape: {df_pandas.shape}")
print(f"Primeras columnas: {list(df_pandas.columns[:10])}")
df_pandas.head(3)

✓ Shape: (6220, 83)
Primeras columnas: ['order', 'Date', 'ST.', 'PO4-P', 'DTP', 'T-P', 'NH4-N', 'NO3+NO2-N', 'NO2-N', 'NO3-N']


/local_disk0/.ephemeral_nfs/envs/pythonEnv-caed0e06-434f-47bc-85a4-b3ce9cb952cd/lib/python3.12/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,order,Date,ST.,PO4-P,DTP,T-P,NH4-N,NO3+NO2-N,NO2-N,NO3-N,...,B,Ba,Ca,Fe,K,Mg,Mn,Na,Si,Sr
0,1,1977-04-18,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1977-04-18,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1977-04-18,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [0]:
# Celda 5 — Limpiar nombres de columnas y tipos básicos

def clean_col_name(col: str) -> str:
    """Normaliza nombres de columna: espacios → _, caracteres especiales → _."""
    import re
    col = str(col).strip()
    col = re.sub(r'[\s\(\)/\-\.]+', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

# Renombrar columnas
df_pandas.columns = [clean_col_name(c) for c in df_pandas.columns]

# Eliminar filas completamente vacías
df_pandas = df_pandas.dropna(how='all')

# Convertir columna Date a datetime
if 'Date' in df_pandas.columns:
    df_pandas['Date'] = pd.to_datetime(df_pandas['Date'], errors='coerce')

# Convertir columnas numéricas (todas excepto ST. y Date)
non_numeric = ['ST_', 'Date']
numeric_cols = [c for c in df_pandas.columns if c not in non_numeric]
for col in numeric_cols:
    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce')

print(f"Shape normalizado: {df_pandas.shape}")
print(f"Columnas numéricas: {len(numeric_cols)}")
print(f"Registros con Date válida: {df_pandas['Date'].notna().sum()}")

Shape normalizado: (6220, 83)
Columnas numéricas: 82
Registros con Date válida: 6220


## 3. Conversión a Spark y enriquecimiento con metadatos

In [0]:
# Celda 6 — Convertir DataFrame pandas → Spark
from pyspark.sql import functions as F
import datetime

INGESTION_TIMESTAMP = datetime.datetime.now(datetime.timezone.utc).isoformat()
DATASET_VERSION     = "v20250925"
SOURCE_URL          = "https://db.cger.nies.go.jp/gem/moni-e/inter/GEMS/database/kasumi/contents/datalist.html"

df_spark = spark.createDataFrame(df_pandas)

df_spark = (
    df_spark
    .withColumn("_ingestion_ts",    F.lit(INGESTION_TIMESTAMP))
    .withColumn("_dataset_version", F.lit(DATASET_VERSION))
    .withColumn("_source_url",      F.lit(SOURCE_URL))
    .withColumn("_source_file",     F.lit("2_water quality_eng_v20250925.xlsx"))
)

print(f"Schema Spark ({len(df_spark.columns)} columnas):")
df_spark.printSchema()

Schema Spark (87 columnas):
root
 |-- order: long (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- ST: long (nullable = true)
 |-- PO4_P: double (nullable = true)
 |-- DTP: double (nullable = true)
 |-- T_P: double (nullable = true)
 |-- NH4_N: double (nullable = true)
 |-- NO3+NO2_N: double (nullable = true)
 |-- NO2_N: double (nullable = true)
 |-- NO3_N: double (nullable = true)
 |-- DON: double (nullable = true)
 |-- DTN: double (nullable = true)
 |-- T_N: double (nullable = true)
 |-- D_COD: double (nullable = true)
 |-- T_COD: double (nullable = true)
 |-- Chl_a: double (nullable = true)
 |-- Pyco: double (nullable = true)
 |-- Phaeo: double (nullable = true)
 |-- SS: double (nullable = true)
 |-- POC: double (nullable = true)
 |-- PON: double (nullable = true)
 |-- C_N: double (nullable = true)
 |-- DOC: double (nullable = true)
 |-- Het_B: double (nullable = true)
 |-- TB: double (nullable = true)
 |-- GP: double (nullable = true)
 |-- NP: double (nullable = true)


## 4. Escritura en Delta Lake (capa Bronze)

In [0]:
# Celda 7 — Crear esquema si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Esquema asegurado: {CATALOG}.{SCHEMA}")

Esquema asegurado: workspace.kasumigaura


In [0]:
# Celda 8 — Escribir en Delta Lake (modo overwrite para idempotencia)
(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_BRONZE)
)

n_rows = spark.table(TABLE_BRONZE).count()
n_cols = len(spark.table(TABLE_BRONZE).columns)
print(f"✓ Tabla Bronze escrita: {TABLE_BRONZE}")
print(f"  Filas  : {n_rows:,}")
print(f"  Columnas: {n_cols}")

✓ Tabla Bronze escrita: workspace.kasumigaura.bronze_water_quality
  Filas  : 6,220
  Columnas: 87


## 5. Validación de la ingesta

In [0]:
# Celda 9 — Aserción de integridad: conteo de filas del Excel == Bronze
n_excel = len(df_pandas)
n_delta = spark.table(TABLE_BRONZE).count()

assert n_delta == n_excel, (
    f"Discrepancia de filas: Excel={n_excel}, Bronze={n_delta}"
)
print(f"✓ Integridad verificada: {n_delta:,} registros en Bronze == {n_excel:,} en Excel")

✓ Integridad verificada: 6,220 registros en Bronze == 6,220 en Excel


In [0]:
# Celda 10 — Distribución temporal de registros por año
from pyspark.sql.functions import year, count

print("Distribución temporal (primeros 10 años):")
(
    spark.table(TABLE_BRONZE)
    .withColumn("year", year("Date"))
    .groupBy("year")
    .agg(count("*").alias("n_registros"))
    .orderBy("year")
    .show(10)
)

print(f"\n✅ Notebook 01 completado — Capa Bronze: {TABLE_BRONZE}")
print(f"   Registros ingresados: {n_delta:,}")
print(f"   Período: 1977–2024")
print(f"   Próximo paso: ejecutar 02_silver_calidad.ipynb")

Distribución temporal (primeros 10 años):
+----+-----------+
|year|n_registros|
+----+-----------+
|1977|         90|
|1978|        120|
|1979|        200|
|1980|        240|
|1981|        250|
|1982|        240|
|1983|        150|
|1984|        120|
|1985|        120|
|1986|        120|
+----+-----------+
only showing top 10 rows

✅ Notebook 01 completado — Capa Bronze: workspace.kasumigaura.bronze_water_quality
   Registros ingresados: 6,220
   Período: 1977–2024
   Próximo paso: ejecutar 02_silver_calidad.ipynb
